# 04. Визуализация графа связей

**User Story 4**: Интерактивный граф для демонстрации и анализа.

**Результат**:
- Интерактивный HTML-граф с цветовой кодировкой
- Сводная таблица по кластерам
- Профили ключевых узлов
- Визуализация отдельных кластеров

**Предпосылки**: Запустите `03_graph_analysis.ipynb`.

---

### Легенда

**Узлы (цвет по типу)**:
- 🔵 Синий — юридическое лицо (company)
- 🟢 Зелёный — физическое лицо (individual)
- 🟠 Оранжевый — ИП (sole_proprietor)
- 🔴 Красноватый — shell-компания

**Узлы (размер)**: пропорционален PageRank

**Рёбра (цвет по типу)**:
- Серый — транзакция
- Красный — доверенность
- Зелёный — зарплатный проект
- Фиолетовый — общие сотрудники

---

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

import pandas as pd
from src import config
from src.viz import (
    create_graph_visualization,
    create_cluster_visualization,
    display_summary_table,
    display_node_profile,
)

## Загрузка данных

In [ ]:
# Загружаем анализированный граф и метрики (локальная ФС)
with open(config.OUTPUT_FILTERED_GRAPH_PICKLE, 'rb') as f:
    G = pickle.load(f)

metrics_df = pd.read_parquet(config.OUTPUT_GRAPH_METRICS)
metrics_df = metrics_df.set_index('client_uk') if 'client_uk' in metrics_df.columns else metrics_df

cluster_summary = pd.DataFrame()
if os.path.exists(config.OUTPUT_CLUSTERS):
    cluster_summary = pd.read_parquet(config.OUTPUT_CLUSTERS)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Metrics: {len(metrics_df)} rows')
print(f'Clusters: {len(cluster_summary)} rows')

## Полный граф

In [ ]:
output_html = os.path.join(config.DATA_DIR, 'graph.html')
net = create_graph_visualization(G, output_file=output_html)
net.show(output_html)

## Сводная таблица по кластерам

In [ ]:
display_summary_table(cluster_summary)

## Визуализация топ-кластеров

In [ ]:
# Визуализация топ-3 кластеров по обороту
if len(cluster_summary) > 0:
    top_clusters = cluster_summary.nlargest(3, 'total_internal_turnover')
    
    for _, row in top_clusters.iterrows():
        cid = int(row['cluster_id'])
        print(f"\n{'='*50}")
        print(f"Cluster {cid}: {row['lead_company_name']}")
        print(f"Members: {int(row['member_count'])}, Turnover: {row['total_internal_turnover']}")
        print('='*50)
        
        cluster_html = os.path.join(config.DATA_DIR, f'cluster_{cid}.html')
        cluster_net = create_cluster_visualization(G, cid, output_file=cluster_html)
        if cluster_net:
            cluster_net.show(cluster_html)
else:
    print('No clusters to visualize.')

## Профиль seed-компании

In [ ]:
# Найти seed-компанию (hop_distance=0)
seed_nodes = [n for n, d in G.nodes(data=True) if d.get('hop_distance', -1) == 0]

if seed_nodes:
    for seed in seed_nodes:
        print(f'Seed company profile: {G.nodes[seed].get("name", seed)}')
        display_node_profile(G, seed, metrics_df)
else:
    print('No seed node found (hop_distance=0). Enter client_uk manually:')
    # display_node_profile(G, YOUR_CLIENT_UK_HERE, metrics_df)

## Профили shell-компаний

In [ ]:
if 'shell_score' in metrics_df.columns:
    shells = metrics_df[metrics_df['shell_score'] >= config.SHELL_SCORE_THRESHOLD]
    shells = shells.sort_values('shell_score', ascending=False)
    
    if len(shells) > 0:
        print(f'Shell companies: {len(shells)}')
        for node_id in shells.index[:5]:  # Top 5
            display_node_profile(G, node_id, metrics_df)
    else:
        print('No shell companies flagged.')
else:
    print('Shell scores not computed. Run 03_graph_analysis.ipynb first.')

---

**Анализ завершён.** Интерактивные HTML-файлы сохранены в `data/`.